In [3]:
from datasets import load_dataset_builder, load_dataset, get_dataset_split_names
from log_wrapper import log_calls

import pandas as pd

/Users/vladpalamarchuk/anaconda3/envs/dev/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ds = load_dataset("yandex/yambda", "flat-multievent-50m", split="train")

# Если нужен pandas DataFrame, можно конвертировать
df = ds.to_pandas()

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47790449 entries, 0 to 47790448
Data columns (total 7 columns):
 #   Column                Dtype  
---  ------                -----  
 0   uid                   uint32 
 1   timestamp             uint32 
 2   item_id               uint32 
 3   is_organic            uint8  
 4   played_ratio_pct      float64
 5   track_length_seconds  float64
 6   event_type            object 
dtypes: float64(2), object(1), uint32(3), uint8(1)
memory usage: 1.6+ GB


## Описание датасета Yandex Yambda

**Yambda** - это бенчмарк для рекомендательных систем на основе реальных данных Яндекс.Музыки.

### Что внутри:
- **Конфигурация**: `flat-multievent-50m` 
- **Размер**: 50 миллионов пользовательских событий
- **Формат**: все типы событий в одной таблице (flat format)

### Колонки в датасете:
1. **uid** - анонимизированный идентификатор пользователя
2. **timestamp** - временная метка события
3. **item_id** - идентификатор музыкального трека
4. **is_organic** - флаг органического взаимодействия (не из рекомендаций)
5. **played_ratio_pct** - процент прослушанного трека (0-100)
6. **track_length_seconds** - длительность трека в секундах
7. **event_type** - тип события:
   - прослушивания (listen)
   - лайки (like)
   - дизлайки (dislike)
   - отмены лайков (unlike)
   - отмены дизлайков (undislike)

In [7]:
df.head()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
0,100,39420,8326270,0,100.0,170.0,listen
1,100,39420,1441281,0,100.0,105.0,listen
2,100,39625,286361,0,100.0,185.0,listen
3,100,40110,732449,0,100.0,240.0,listen
4,100,40360,3397170,0,46.0,130.0,listen


In [9]:
print(f"""
Распределение типов событий:
{df['event_type'].value_counts()}

Уникальных пользователей: {df['uid'].nunique():,}
Уникальных треков: {df['item_id'].nunique():,}

Доля органических взаимодействий: {df['is_organic'].mean():.2%}
""")


Распределение типов событий:
event_type
listen       46467212
like           881456
unlike         312972
dislike        107776
undislike       21033
Name: count, dtype: int64

Уникальных пользователей: 10,000
Уникальных треков: 934,057

Доля органических взаимодействий: 52.05%



## Дополнительные файлы в датасете Yambda

Помимо событий, датасет содержит:

### Маппинги (в корне data/):
- **artist_item_mapping.parquet** - связь исполителей и треков (artist_id, item_id)
- **album_item_mapping.parquet** - связь альбомов и треков (album_id, item_id)
- **embeddings.parquet** - аудиоэмбеддинги треков (item_id, embed, normalized_embed)

### Разделённые по типам событий (в data/flat/5b/, 500m/, 50m/):
- **listens.parquet** - только прослушивания (с метриками played_ratio_pct, track_length_seconds)
- **likes.parquet** - только лайки
- **dislikes.parquet** - только дизлайки
- **unlikes.parquet** - отмены лайков
- **undislikes.parquet** - отмены дизлайков
- **multi_event.parquet** - все события вместе (то, что вы загрузили как "flat-multievent-50m")

### Последовательные форматы (в data/sequential/):
- Данные, сгруппированные по пользователям в виде последовательностей

### Версии по размеру:
- **5B** - 5 миллиардов событий (полный датасет)
- **500M** - 500 миллионов событий
- **50M** - 50 миллионов событий (ваша версия)

## Бенчмарки и задачи

Датасет используется для двух основных задач рекомендаций:

### 1. Listen+ (предсказание прослушиваний)
Цель: рекомендовать треки, которые пользователь прослушает более чем на 50%

**Лучшие результаты на Yambda-50M** (по Recall@100):
- **SASRec**: 0.1028 - секвенциальная модель с самовниманием
- **ItemKNN**: 0.1297 - k-ближайших соседей по айтемам
- **BPR**: 0.0836 - Bayesian Personalized Ranking

### 2. Like (предсказание лайков)
Цель: рекомендовать треки, которым пользователь поставит лайк

**Лучшие результаты на Yambda-50M** (по Recall@100):
- **SASRec**: 0.0679 
- **ItemKNN**: 0.0648
- **DecayPop**: 0.0651 - популярность с временным затуханием

### Метрики:
- **NDCG@10/100** - качество ранжирования топ-10/100 рекомендаций
- **Recall@10/100** - доля релевантных айтемов в топ-10/100
- **Coverage@10/100** - покрытие каталога рекомендациями

In [ ]:
# Как загрузить другие части датасета:

# Только прослушивания
# ds_listens = load_dataset("yandex/yambda", "flat-listens-50m")

# Только лайки
# ds_likes = load_dataset("yandex/yambda", "flat-likes-50m")

# Последовательный формат (для секвенциальных моделей)
# ds_seq = load_dataset("yandex/yambda", "sequential-50m")

# Эмбеддинги треков
# ds_embeddings = load_dataset("yandex/yambda", "embeddings")

# Маппинги исполнителей
# ds_artists = load_dataset("yandex/yambda", "artist_item_mapping")

print("Доступные конфигурации датасета Yambda")